In [ ]:
import os
import random
import hashlib
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt


# -----------------------------
# Config
# -----------------------------
IMAGE_DIR = "/Users/juliangendreau/Desktop/mgmt_data_aligned"
CSV_PATH = "/Users/juliangendreau/Desktop/labels.csv"

TARGET_SHAPE = (160, 160, 160)
BATCH_SIZE = 2
LR = 1e-5                     # LOWER LR TO PREVENT NaNs
EPOCHS = 50
VAL_SPLIT = 0.2
RANDOM_SEED = 42
NUM_WORKERS = 0

USE_CACHE = True
CACHE_DIR = "./mgmt_cache_160"

EARLY_STOP_PATIENCE = 5

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------
# Utility functions
# -----------------------------
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def resample_volume(volume, target_shape):
    zoom_factors = [t / c for t, c in zip(target_shape, volume.shape)]
    return zoom(volume, zoom_factors, order=1)


def clip_and_normalize(volume):
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

    lo = np.percentile(volume, 1)
    hi = np.percentile(volume, 99)
    volume = np.clip(volume, lo, hi)

    mean = volume.mean()
    std = volume.std()

    if std < 1e-6:
        std = 1e-6

    return (volume - mean) / std


def cache_key(pid, target_shape):
    s = f"{pid}_{target_shape}_nocrop"
    return hashlib.md5(s.encode("utf-8")).hexdigest() + ".npy"


def show_middle_slice(volume, title="Middle Slice"):
    vol_np = volume.squeeze(0).cpu().numpy()
    mid = vol_np.shape[0] // 2
    plt.figure(figsize=(4, 4))
    plt.imshow(vol_np[mid], cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()


# -----------------------------
# Dataset (MRI + clinical)
# -----------------------------
class MGMTDataset(Dataset):
    def __init__(self, image_dir, labels_df, target_shape,
                 use_cache=False, cache_dir=None):
        self.image_dir = image_dir
        self.labels_df = labels_df.reset_index(drop=True)
        self.target_shape = target_shape
        self.use_cache = use_cache
        self.cache_dir = cache_dir

        if self.use_cache:
            ensure_dir(self.cache_dir)

    def __len__(self):
        return len(self.labels_df)

    def _load_and_preprocess(self, pid):
        nii_path = os.path.join(self.image_dir, f"{pid}.nii.gz")
        if not os.path.exists(nii_path):
            nii_path = os.path.join(self.image_dir, f"{pid}.nii")
            if not os.path.exists(nii_path):
                raise FileNotFoundError(f"Missing NIfTI for patient_id {pid}")

        img = nib.load(nii_path).get_fdata().astype(np.float32)
        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

        if img.ndim != 3:
            raise ValueError(f"Scan for {pid} has shape {img.shape}, expected 3D")

        img = clip_and_normalize(img)
        img = resample_volume(img, self.target_shape)

        return img

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        pid = str(row["patient_id"])
        label = int(row["mgmt_methylated"])

        # Clinical variables
        age = row.get("age", 0.0)
        sex = row.get("sex_x", 0.0)

        # Convert to float safely
        try:
            age = float(age)
        except:
            age = 0.0

        try:
            sex = float(sex)
        except:
            sex = 0.0

        # Replace NaN with 0
        if np.isnan(age):
            age = 0.0
        if np.isnan(sex):
            sex = 0.0

        # Normalize age
        age = (age - 50.0) / 20.0

        if self.use_cache:
            key = cache_key(pid, self.target_shape)
            cache_path = os.path.join(self.cache_dir, key)
            if os.path.exists(cache_path):
                img = np.load(cache_path)
            else:
                img = self._load_and_preprocess(pid)
                np.save(cache_path, img)
        else:
            img = self._load_and_preprocess(pid)

        img = torch.from_numpy(img).unsqueeze(0)
        clinical = torch.tensor([age, sex], dtype=torch.float32)

        return img, clinical, torch.tensor(label, dtype=torch.long)


# -----------------------------
# Efficient3D-Medium (with clinical fusion)
# -----------------------------
class ConvBNAct3D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size, stride=stride,
                              padding=padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm3d(out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Conv3d(channels, hidden, 1)
        self.fc2 = nn.Conv3d(hidden, channels, 1)
        self.act = nn.SiLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        s = self.pool(x)
        s = self.act(self.fc1(s))
        s = self.sigmoid(self.fc2(s))
        return x * s


class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=4, stride=1, kernel_size=3):
        super().__init__()
        mid = in_ch * expand_ratio
        self.use_res = (stride == 1 and in_ch == out_ch)

        layers = []
        if expand_ratio != 1:
            layers.append(ConvBNAct3D(in_ch, mid, 1))

        layers.append(ConvBNAct3D(mid, mid, kernel_size, stride, kernel_size // 2, groups=mid))
        layers.append(SEBlock3D(mid))
        layers.append(nn.Conv3d(mid, out_ch, 1, bias=False))
        layers.append(nn.BatchNorm3d(out_ch))

        self.block = nn.Sequential(*layers)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        out = self.block(x)
        if self.use_res:
            out = out + x
        return self.act(out)


class Efficient3DMedium(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.stem = ConvBNAct3D(1, 24, 3, 2, 1)
        self.blocks = nn.Sequential(
            MBConv3D(24, 24, 1, 1),
            MBConv3D(24, 32, 4, 2),
            MBConv3D(32, 32, 4, 1),
            MBConv3D(32, 48, 4, 2),
            MBConv3D(48, 48, 4, 1),
            MBConv3D(48, 64, 4, 2),
            MBConv3D(64, 64, 4, 1),
        )
        self.pool = nn.AdaptiveAvgPool3d(1)

        # Clinical fusion: 64 MRI features + 2 clinical
        self.fc1 = nn.Linear(64 + 2, 32)
        self.act = nn.SiLU(inplace=True)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x, clinical):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)

        x = torch.cat([x, clinical], dim=1)

        x = self.act(self.fc1(x))
        return self.fc2(x)


# -----------------------------
# Metrics
# -----------------------------
def compute_auc(labels, probs):
    try:
        return roc_auc_score(labels, probs)
    except:
        return float("nan")


# -----------------------------
# Training + Early Stopping
# -----------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, scaler, epoch):
    model.train()
    total_loss, total_correct, total = 0, 0, 0
    all_labels, all_probs = [], []

    first_batch = True

    for imgs, clinical, labels in loader:
        if first_batch:
            show_middle_slice(imgs[0], title=f"Epoch {epoch+1} - Example Slice")
            first_batch = False

        imgs, clinical, labels = imgs.to(device), clinical.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(imgs, clinical)
            loss = criterion(outputs, labels)

        if torch.isnan(loss):
            print("NaN detected in training loss — skipping batch")
            continue

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

        probs = torch.softmax(outputs, 1)[:, 1]
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    return (
        total_loss / max(total, 1),
        total_correct / max(total, 1),
        compute_auc(all_labels, all_probs),
    )


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for imgs, clinical, labels in loader:
            imgs, clinical, labels = imgs.to(device), clinical.to(device), labels.to(device)
            outputs = model(imgs, clinical)
            loss = criterion(outputs, labels)

            if torch.isnan(loss):
                print("NaN detected in validation loss — skipping batch")
                continue

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)
            total_correct += (preds == labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(outputs, 1)[:, 1]
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (
        total_loss / max(total, 1),
        total_correct / max(total, 1),
        compute_auc(all_labels, all_probs),
    )


# -----------------------------
# Main
# -----------------------------
def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    labels_df = pd.read_csv(CSV_PATH)
    labels_df["patient_id"] = labels_df["patient_id"].astype(str)
    y = labels_df["mgmt_methylated"].values

    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=VAL_SPLIT, random_state=RANDOM_SEED
    )
    train_idx, val_idx = next(splitter.split(np.zeros(len(y)), y))

    train_df = labels_df.iloc[train_idx].reset_index(drop=True)
    val_df = labels_df.iloc[val_idx].reset_index(drop=True)

    train_dataset = MGMTDataset(
        IMAGE_DIR, train_df, TARGET_SHAPE,
        use_cache=USE_CACHE, cache_dir=CACHE_DIR
    )
    val_dataset = MGMTDataset(
        IMAGE_DIR, val_df, TARGET_SHAPE,
        use_cache=USE_CACHE, cache_dir=CACHE_DIR
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Efficient3DMedium().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    best_auc = -1
    patience_counter = 0

    print(f"Training samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    for epoch in range(EPOCHS):
        train_loss, train_acc, train_auc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler, epoch
        )
        val_loss, val_acc, val_auc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch+1}/{EPOCHS} "
            f"- Train loss: {train_loss:.4f}, acc: {train_acc:.4f}, AUC: {train_auc:.4f} "
            f"- Val loss: {val_loss:.4f}, acc: {val_acc:.4f}, AUC: {val_auc:.4f}"
        )

        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print("Early stopping triggered.")
                break

    print("Training complete. Best model saved as best_model.pth")


if __name__ == "__main__":
    main()
